In [1]:
# train_kfold.py
from utils.utils import *
from scripts.config import build_cv_splits, build_model_cfg_from_dataloader  # <- only import helpers
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
import wandb
from datetime import datetime

from datautils.midas import class_coverage

import argparse
import warnings
import os
import time
import json
from typing import Dict, Any, List

try:
    import yaml  # optional, for YAML configs
except Exception:
    yaml = None

warnings.filterwarnings("ignore", message="Accurate seek is not implemented for pyav backend")
torch.manual_seed(0)


def is_number(x):
    try:
        float(x)
        return True
    except Exception:
        return False


def flatten_numeric(d: Dict[str, Any]) -> Dict[str, float]:
    out = {}
    def _walk(prefix, obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                _walk(f"{prefix}.{k}" if prefix else k, v)
        else:
            if isinstance(obj, (int, float)) or (hasattr(obj, "item") and getattr(obj, "dim", lambda:1)() == 0):
                try:
                    out[prefix] = float(obj if not hasattr(obj, "item") else obj.item())
                except Exception:
                    pass
    _walk("", d)
    return out


def write_json(path: str, obj: Any):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def write_csv(path: str, rows: List[Dict[str, Any]]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    headers = []
    for r in rows:
        for k in r.keys():
            if k not in headers:
                headers.append(k)
    with open(path, "w") as f:
        f.write(",".join(headers) + "\n")
        for r in rows:
            f.write(",".join(str(r.get(h, "")) for h in headers) + "\n")


def load_config_file(path: str) -> Dict[str, Any]:
    ext = os.path.splitext(path)[1].lower()
    with open(path, "r") as f:
        if ext in (".yml", ".yaml"):
            if yaml is None:
                raise RuntimeError("pyyaml not installed; install it or use JSON.")
            return yaml.safe_load(f)
        elif ext == ".json":
            return json.load(f)
        else:
            raise ValueError(f"Unsupported config extension: {ext} (use .yaml/.yml or .json)")


class DefaultArgsNamespace:
    """
    A minimal args container built from a config dict that contains at least:
      - dataloader_params
      - learning_params
      - transformer_params (optional)
      - tcn_model_params (optional)
      - model_cfg_overrides (optional)  # to override d_model, nhead, etc.
    """
    def __init__(self, cfg: Dict[str, Any], fold_index: int = 0):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 1) take dataloader params from file (no global imports)
        self.dataloader_params: Dict[str, Any] = dict(cfg["dataloader_params"])

        # 2) build CV folds from the loaded dataloader params
        folds = build_cv_splits(self.dataloader_params)
        if not folds:
            raise RuntimeError("No valid CV folds were produced from the provided dataloader_params['all_trials'] and ['cv'].")

        if fold_index < 0 or fold_index >= len(folds):
            raise IndexError(f"fold_index {fold_index} out of range [0, {len(folds)-1}]")
        self.fold = folds[fold_index]

        # inject chosen fold back
        self.dataloader_params["train_trials"] = self.fold["train_trials"]
        self.dataloader_params["val_trials"]   = self.fold["val_trials"]
        self.dataloader_params["test_trials"]  = self.fold["test_trials"]

        base_exp = self.dataloader_params.get("experiment_name", "exp")
        self.dataloader_params["experiment_name"] = f"{base_exp}_{self.fold['name']}"

        # 3) model cfg derived from dataloader selections (and optional overrides)
        overrides = cfg.get("model_cfg_overrides", {})
        self.mmtransformercfg = build_model_cfg_from_dataloader(self.dataloader_params, **overrides)

        # 4) the rest
        self.learning_params     = dict(cfg["learning_params"])
        self.transformer_params  = dict(cfg.get("transformer_params", {}))
        self.tcn_model_params    = dict(cfg.get("tcn_model_params", {}))
        self.record_results      = bool(cfg.get("record_results", True))



In [3]:
# JUPYTER-FRIENDLY: no argparse; hardcoded defaults + safe W&B

import os, json, time, pathlib
from types import SimpleNamespace

# optional deps (safe in notebooks)
try:
    import yaml
except Exception:
    yaml = None

try:
    import wandb
except Exception:
    class _W:
        @staticmethod
        def init(**kwargs):
            class _Dummy:
                def log(self, *a, **k): pass
                def finish(self): pass
            print("[wandb] not installed; running with a dummy logger.")
            return _Dummy()
    wandb = _W()

# ----- helpers: lightweight config loader -----
def load_config_file(path):
    path = os.path.abspath(path)
    with open(path, "r") as f:
        if path.endswith((".yml", ".yaml")):
            if yaml is None:
                raise RuntimeError("pyyaml not installed; cannot read YAML config.")
            return yaml.safe_load(f)
        elif path.endswith(".json"):
            return json.load(f)
        else:
            raise ValueError(f"Unsupported config extension: {path}")

# NOTE: assumes you already have these in your environment/codebase:
# - build_cv_splits(cfg["dataloader_params"])
# - DefaultArgsNamespace(cfg, fold_index=fi)
# - MIDAS_get_dataloaders(args)
# - print_one_batch(loader)
# - class_coverage(train_class_stats, test_class_stats)

# ---------------- HARD-CODED "CLI" VALUES ----------------
cmd_args = SimpleNamespace(
    config = "./configs/RAVEN_DATA/exp1_window_10hz.yaml",  # <— change if needed
    job_id = None,                                   # auto-filled below
    wandb  = "off",                                  # "on" or "off"
    fold_index = None,                               # e.g., 0 to run a single fold
)

# ---------------- LOAD CONFIG ----------------
cfg = load_config_file(cmd_args.config)
print("Loaded config from:", os.path.abspath(cmd_args.config))
print(cfg)

# ---------------- JOB/OUTPUT DIRS ----------------
if cmd_args.job_id is None:
    cmd_args.job_id = str(int(time.time()))
base_job_id = "MTRSAP_" + cmd_args.job_id

# build folds (for info / selection)
folds = build_cv_splits(cfg["dataloader_params"])
print(f"Discovered {len(folds)} folds:")
for i, f in enumerate(folds):
    print(f"  [{i}] {f['name']}: train={f['train_trials']}  val={f['val_trials']}  test={f['test_trials']}")

fold_indices = list(range(len(folds))) if cmd_args.fold_index is None else [cmd_args.fold_index]

run_root = os.path.abspath(f'./results/job_{base_job_id}')
ckpt_root = os.path.abspath(f'./checkpoints/job_{base_job_id}')
os.makedirs(run_root, exist_ok=True)
os.makedirs(ckpt_root, exist_ok=True)

# snapshot exact config used
snap_cfg_path = os.path.join(run_root, "config_used" + os.path.splitext(cmd_args.config)[1])
if not os.path.exists(snap_cfg_path):
    with open(snap_cfg_path, "w") as f:
        if snap_cfg_path.endswith((".yml", ".yaml")) and yaml is not None:
            yaml.safe_dump(cfg, f, sort_keys=False)
        else:
            json.dump(cfg, f, indent=2)
print("Wrote config snapshot to:", snap_cfg_path)

# ---------------- MAIN LOOP (per fold) ----------------
from typing import List, Dict, Any

fold_rows_for_csv: List[Dict[str, Any]] = []
numeric_metrics_per_key: Dict[str, List[float]] = {}

for fi in fold_indices:
    print(f"\n=== Running Fold {fi}: {folds[fi]['name']} ===")
    args = DefaultArgsNamespace(cfg, fold_index=fi)
    experiment_name = args.dataloader_params['experiment_name']

    # per-fold dirs
    fold_results_dir = os.path.join(run_root, experiment_name)
    fold_ckpt_dir = os.path.join(ckpt_root, experiment_name)
    os.makedirs(fold_results_dir, exist_ok=True)
    os.makedirs(fold_ckpt_dir, exist_ok=True)

    print(f"Fold results dir: {fold_results_dir}")
    print(f"Fold checkpoints dir: {fold_ckpt_dir}")

    # W&B (safe in notebook; will be disabled unless cmd_args.wandb == "on")
    wandb_mode = "online" if cmd_args.wandb == "on" else "disabled"
    wandb_logger = wandb.init(
        project="MIDAS Gesture Recognition",
        group=f"Gesture Recognition ({base_job_id})",
        mode=wandb_mode,
        name=experiment_name,
        notes=f"job={base_job_id}, fold={folds[fi]['name']}",
        config={"args": str(args.dataloader_params)},
    )

    keysteps   = args.dataloader_params['keysteps']
    out_classes = len(keysteps)
    modality   = args.dataloader_params['modalities']
    selections = args.dataloader_params['selections']

    print(f"Keysteps: {keysteps}")
    print(f"Modalities: {modality}")
    print(f"Selections: {selections}")
    print(f"Trials (train/val/test): {args.dataloader_params['train_trials']} / "
          f"{args.dataloader_params['val_trials']} / {args.dataloader_params['test_trials']}")

    # ----- Data -----
    train_loader, val_loader, test_loader, train_class_stats, val_class_stats, test_class_stats = MIDAS_get_dataloaders(args)
    args.dataloader_params['train_class_stats'] = train_class_stats
    args.dataloader_params['val_class_stats'] = val_class_stats

    print(f"Training samples: {len(train_loader.dataset)}, "
          f"Validation samples: {len(val_loader.dataset)}, "
          f"Test samples: {len(test_loader.dataset)}")

    print_one_batch(train_loader)

    # coverage check (avoid impossible folds)
    # missing = class_coverage(train_class_stats, test_class_stats)
    # print(f"Classes missing in training set (but present in test set): {missing}")

    # >>> Your training/eval code for this fold goes here <<<
    # e.g., train_one_fold(args, train_loader, val_loader, test_loader, fold_results_dir, fold_ckpt_dir, wandb_logger)

print("\nAll requested folds finished.")


Loaded config from: /sfs/gpfs/tardis/home/cjh9fw/Desktop/2025/DataCollectionSystem/benchmarks/gesture_recognition/MTRSAP/configs/RAVEN_DATA/exp1_window_10hz.yaml
{'record_results': True, 'dataloader_params': {'experiment_name': 'hamid_resnet_DS_NOCLUTCH_NOWINDOW_only', 'base_path': '/standard/UVA-DSA/MIDAS/Organized/final_data/', 'batch_size': 4, 'sample_rate': 10, 'ignore_clutch': True, 'clutch_pressed_value': 0, 'all_trials': ['t1', 't3', 't4', 't5', 't6', 't7'], 'cv': {'scheme': 'leave_one_out', 'k': 5, 'val_ratio': 0.25, 'seed': 42, 'shuffle': True, 'val_same_as_test': True}, 'modalities': ['images'], 'selections': {'images': ['resnet']}, 'observation_window': 10, 'step': 1, 'keysteps': {'S1': 'Approach peg', 'S2': 'Align & grasp', 'S3': 'Lift peg', 'S4': 'Transfer peg - Get together', 'S5': 'Transfer peg - Exchange', 'S6': 'Approach pole', 'S7': 'Align & place'}}, 'learning_params': {'lr': 1e-05, 'epochs': 100, 'weight_decay': 1e-05, 'patience': 30, 'lr_drop': 30, 'best_chkpoint':